In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [34]:
!pip install -Uq langchain_experimental
from langchain_experimental.text_splitter import SemanticChunker

In [45]:
!pip install -Uq langchain_chroma
!pip install -Uq langchain langchain-community langchain-openai
!pip install -Uq python_dotenv
!pip install -Uq "unstructured[all-docs]"
!pip install -Uq langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.4/719.4 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.5/236.5 kB 25.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.43.0, but you have google-auth 2.48.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.39.1 which is incompatible.
google-adk 1.21.0 requires opentelemetry-api<=1.37.0,>=1.37.0, but you have opentelemetry-api 1.39.1 which is incompatible.
google-adk 1.21.0 requires ope

In [46]:

import os
from langchain_community.document_loaders import TextLoader, DirectoryLoader, UnstructuredFileLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv


load_dotenv()

False

In [47]:
data_path = "/content/drive/MyDrive/IITSTUFF" # UPDATE THIS PATH to your data file or directory

In [48]:
def load_documents(docs_path= data_path):
    """Load all document files from the specified path."""
    print(f"Loading documents from {docs_path}...")

    # Check if docs directory exists
    if not os.path.exists(docs_path):
        raise FileNotFoundError(f"The directory {docs_path} does not exist. Please create it and add your company files.")

    # Load all .pdf files from the docs directory using UnstructuredFileLoader
    loader = DirectoryLoader(
        path=docs_path,
        glob="*.pdf", # Changed to look for PDF files
        loader_cls=UnstructuredFileLoader # Changed to UnstructuredFileLoader
    )

    documents = loader.load()

    if len(documents) == 0:
        raise FileNotFoundError(f"No .pdf files found in {docs_path}. Please add your company documents.")


    for i, doc in enumerate(documents[:2]):  # Show first 2 documents
        print(f"\nDocument {i+1}:")
        print(f"  Source: {doc.metadata['source']}")
        print(f"  Content length: {len(doc.page_content)} characters")
        print(f"  Content preview: {doc.page_content[:100]}...")
        print(f"  metadata: {doc.metadata}")

    return documents

In [58]:
from langchain_core.documents import Document

def split_documents(documents, chunk_size=1000, chunk_overlap=0):
    """Split documents into smaller chunks with overlap using semantic chunking."""
    print("Splitting documents into chunks using semantic chunking...")

    # Concatenate all page_content into a single string for semantic chunking
    full_text = " ".join([doc.page_content for doc in documents])

    semantic_splitter = SemanticChunker(
        embeddings=GoogleGenerativeAIEmbeddings(model="models/text-embedding-004"),
        breakpoint_threshold_type="percentile",  # or "standard_deviation"
        breakpoint_threshold_amount=70
    )

    string_chunks = semantic_splitter.split_text(full_text)

    # Convert string chunks back into Document objects
    document_chunks = [Document(page_content=chunk) for chunk in string_chunks]

    print("SEMANTIC CHUNKING RESULTS:")
    print("=" * 50)
    for i, chunk in enumerate(document_chunks, 1):
        print(f"Chunk {i}: ({len(chunk.page_content)} chars)")
        print(f'"{chunk.page_content}"')
        print()
    return document_chunks

In [59]:
def create_vector_store(chunks, persist_directory="db/chroma_db"):
    """Create and persist ChromaDB vector store"""
    print("Creating embeddings and storing in ChromaDB...")

    embedding_model = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

    # Create ChromaDB vector store
    print("--- Creating vector store ---")
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        persist_directory=persist_directory,
        collection_metadata={"hnsw:space": "cosine"}
    )
    print("--- Finished creating vector store ---")

    print(f"Vector store created and saved to {persist_directory}")
    return vectorstore

In [60]:
# Create a .env file and add your OpenAI API key
from google.colab import userdata

# It's recommended to store your API key securely in Colab Secrets
# Go to the '🔑' icon on the left panel, click 'Add new secret', set name as 'OPENAI_API_KEY' and paste your key.
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# Verify the key is loaded (optional)
if os.getenv("GOOGLE_API_KEY"):
    print("Gemini API key loaded successfully.")
else:
    print("OpenAI API key not found. Please add it to Colab Secrets as 'OPENAI_API_KEY'.")

Gemini API key loaded successfully.


In [61]:
def main():
    """Main ingestion pipeline"""
    print("=== RAG Document Ingestion Pipeline ===\n")

    # Define paths
    docs_path = data_path
    persistent_directory = "db/chroma_db"

    # Check if vector store already exists
    if os.path.exists(persistent_directory):
        print("✅ Vector store already exists. No need to re-process documents.")

        embedding_model = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
        vectorstore = Chroma(
            persist_directory=persistent_directory,
            embedding_function=embedding_model,
            collection_metadata={"hnsw:space": "cosine"}
        )
        print(f"Loaded existing vector store with {vectorstore._collection.count()} documents")
        return vectorstore

    print("Persistent directory does not exist. Initializing vector store...\n")

    # Step 1: Load documents
    documents = load_documents(docs_path)

    # Step 2: Split into chunks
    chunks = split_documents(documents)

    # # Step 3: Create vector store
    vectorstore = create_vector_store(chunks, persistent_directory)

    print("\n✅ Ingestion complete! Your documents are now ready for RAG queries.")
    return vectorstore


if __name__ == "__main__":
    main()

=== RAG Document Ingestion Pipeline ===

Persistent directory does not exist. Initializing vector store...

Loading documents from /content/drive/MyDrive/IITSTUFF...

Document 1:
  Source: /content/drive/MyDrive/IITSTUFF/Student Handbook v2.pdf
  Content length: 86745 characters
  Content preview: Bachelor of Science [Hons] Artificial Intelligence and Data Science

Student Handbook [Version 1.0]
...
  metadata: {'source': '/content/drive/MyDrive/IITSTUFF/Student Handbook v2.pdf'}
Splitting documents into chunks using semantic chunking...
SEMANTIC CHUNKING RESULTS:
Chunk 1: (1984 chars)
"Bachelor of Science [Hons] Artificial Intelligence and Data Science

Student Handbook [Version 1.0]

Department of Computer Science and Engineering

INFORMATICS INSTITUTE OF TECHNOLOGY SRI LANKA

Validation: 04 May 2020 Revision Date:

School of Computing Science and Digital Media

ROBERT GORDON UNIVERSITY ABERDEEN, SCOTLAND, UNITED KINGDOM

THIS PAGE IS INTENTIONALLY LEFT BLANK

BSc [Hons] Artificial I